# Lesson 06 — Cost of Friction Model

## Goal

Quantify the hidden financial cost of slow, manual, fragmented workflows by breaking costs into three friction layers: base manual labor, rework, and expert interruptions.

---

## Learning Objectives

By the end of this lesson, you will:

1. **Understand 3 friction layers** — Base (necessary work) vs Rework (preventable) vs Expert (opportunity cost)
2. **Quantify hidden waste** — Measure €13,933 in friction across 3 workflows (18% of total cost)
3. **Identify high-friction workflows** — Rank by opportunity (% of cost) to prioritize improvements
4. **Connect friction to value** — Understand €13,933 friction is 72% of potential L03 labor savings
5. **Prepare for root cause analysis** — Friction measurement sets up L07 (value stream mapping) and L08 (graph bottleneck detection)

---

## Prerequisites

- Lesson 02: Operating Metrics to Cost Models (cost data, workflow volumes, touch times)
- Lesson 05: PE Value Levers (understand labor productivity as value creation opportunity)

---

## Core Insight

**Friction cost ≠ total cost.** Every workflow has necessary labor (base cost) and preventable waste (friction). Measuring friction surfaces the *opportunity* — before automation design, we diagnose what's actually broken.

## Part 1 — Understanding Friction

### The 3 Friction Layers

Every workflow cost has three components:

#### **Layer 1: Base Manual Cost** (Necessary Work)
The cost of handling work efficiently under current process design.

**Formula:** `volume × touch_time_minutes / 60 × hourly_cost`

**Example:** Invoice approval
- 4,200 invoices/year × 12 minutes × €55/hr = €4,620/year
- This is "necessary" — we still need to review and approve invoices

---

#### **Layer 2: Rework Cost** (Preventable Waste)
The cost of re-doing work due to errors, missing data, or process breakdowns. **This is preventable.**

**Formula:** `volume × rework_rate × rework_minutes / 60 × hourly_cost`

**Example:** Invoice approval
- 5% of invoices arrive with missing fields (GL code, vendor)
- Each rejection requires re-submission + 10-minute reprocessing
- Cost: 4,200 × 5% × 10 min × €55/hr = €231/year
- **This cost vanishes if we improve data quality or pre-fill forms**

---

#### **Layer 3: Expert Interruption Cost** (Opportunity Cost)
The cost of high-skilled people pulled away from their primary work to handle exceptions, escalations, or reviews. **This represents lost opportunity.**

**Formula:** `volume × escalation_rate × expert_minutes / 60 × expert_hourly_cost`

**Example:** Invoice approval
- 2% of invoices are escalated to manager for approval (unusual vendors, large amounts)
- Manager spends 20 minutes per escalation
- Manager hourly cost: €75/hr (higher than analyst €55/hr)
- Cost: 4,200 × 2% × 20 min × €75/hr = €77/year
- **If we could route 95% of invoices automatically, the manager's time becomes available for strategy/growth**

---

### Key Insight: Total Cost = Base + Friction

```
Total Cost = Base (necessary) + Rework (preventable) + Expert Interruption (opportunity)
Friction = Rework + Expert Interruption = Preventable + Opportunity
```

The opportunity for value creation is **Friction cost**, not total cost. If we eliminate friction, the base cost remains (we still need approvers), but €13,933 in waste disappears.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the cost_of_friction function
def cost_of_friction(
    annual_cases,
    manual_minutes,
    rework_rate,
    rework_minutes,
    loaded_hourly_cost,
    expert_interruptions_per_case=0,
    expert_minutes_per_interruption=0,
    expert_hourly_cost=0,
):
    """
    Calculate friction cost (preventable waste + opportunity cost) in a workflow.
    
    Args:
        annual_cases: Annual volume (e.g., 4,200 invoices)
        manual_minutes: Touch time per case (e.g., 12 minutes)
        rework_rate: Percentage of cases needing rework (e.g., 0.05 = 5%)
        rework_minutes: Time to rework each case (e.g., 10 minutes)
        loaded_hourly_cost: Labor rate including benefits (€/hr)
        expert_interruptions_per_case: Percentage of cases escalated (e.g., 0.02 = 2%)
        expert_minutes_per_interruption: Time expert spends per escalation
        expert_hourly_cost: Expert labor rate including benefits (€/hr)
    
    Returns:
        dict with base, rework, expert, total, and friction breakdown
    """
    # Layer 1: Base manual cost (necessary work)
    base = annual_cases * manual_minutes / 60 * loaded_hourly_cost
    
    # Layer 2: Rework cost (preventable waste)
    rework = annual_cases * rework_rate * rework_minutes / 60 * loaded_hourly_cost
    
    # Layer 3: Expert interruption cost (opportunity cost)
    expert = (
        annual_cases
        * expert_interruptions_per_case
        * expert_minutes_per_interruption
        / 60
        * expert_hourly_cost
    )
    
    total = base + rework + expert
    friction = rework + expert
    friction_pct = (friction / total * 100) if total > 0 else 0
    
    return {
        "base_manual_cost": base,
        "rework_cost": rework,
        "expert_interruption_cost": expert,
        "total_cost": total,
        "friction_cost": friction,
        "friction_percentage": friction_pct,
        "cost_per_case": total / annual_cases if annual_cases > 0 else 0,
    }

print("✓ cost_of_friction() function loaded")
print("\nFunction signature:")
print("  cost_of_friction(annual_cases, manual_minutes, rework_rate, rework_minutes,")
print("                   loaded_hourly_cost, expert_interruptions_per_case=0,")
print("                   expert_minutes_per_interruption=0, expert_hourly_cost=0)")

### Recap: 3 Workflows from Lesson 02

We'll analyze the same three workflows from Lessons 02-05 to build consistency:

In [ ]:
# Three workflows from Lesson 02: baseline data
workflows_data = {
    "Invoice Approval": {
        "annual_volume": 4200,
        "touch_time_min": 12,
        "rework_rate": 0.05,  # 5% rejected for missing fields
        "rework_time_min": 10,  # Time to resubmit and reprocess
        "hourly_cost": 55,  # Analyst
        "escalation_rate": 0.02,  # 2% escalated to manager
        "escalation_time_min": 20,  # Manager review time
        "expert_cost": 75,  # Manager rate
    },
    "Support Triage": {
        "annual_volume": 18000,
        "touch_time_min": 14,
        "rework_rate": 0.08,  # 8% misrouted initially
        "rework_time_min": 18,  # Re-triage time
        "hourly_cost": 55,  # Analyst
        "escalation_rate": 0.15,  # 15% escalated to specialist
        "escalation_time_min": 30,  # Specialist investigation
        "expert_cost": 75,  # Specialist rate
    },
    "Management Reporting": {
        "annual_volume": 52,
        "touch_time_min": 180,  # Data gathering + analysis
        "rework_rate": 0.30,  # 30% require revision
        "rework_time_min": 120,  # Major revision cycle
        "hourly_cost": 55,  # Analyst
        "escalation_rate": 0.05,  # 5% escalated for executive review
        "escalation_time_min": 30,  # Executive review
        "expert_cost": 90,  # Executive rate
    },
}

print("3 WORKFLOWS FOR FRICTION ANALYSIS")
print("=" * 70)
for workflow, params in workflows_data.items():
    print(f"\n{workflow}:")
    print(f"  Volume: {params['annual_volume']:,}/year")
    print(f"  Touch time: {params['touch_time_min']} min | Rework: {params['rework_rate']*100:.0f}% | Escalation: {params['escalation_rate']*100:.0f}%")
    print(f"  Analyst cost: €{params['hourly_cost']}/hr | Expert cost: €{params['expert_cost']}/hr")

---

## Part 2 — Worked Example 1: Invoice Approval

### Analysis: Where is Friction in Invoice Approval?

**Workflow:**
- 4,200 invoices per year
- Analyst reviews each: 12 minutes (check amount, vendor, GL code)
- **Friction point 1 — Rework (5%):** Some invoices arrive with missing GL code → rejection → resubmission
  - 5% × 10 minutes per rework
- **Friction point 2 — Expert escalation (2%):** Unusual vendors or large amounts → escalated to manager for approval
  - 2% × 20 minutes manager time per escalation

**Question:** How much of the €4,928 total cost (from Lesson 02) is friction vs necessary work?

In [ ]:
# Worked Example 1: Invoice Approval Friction Analysis
invoice_friction = cost_of_friction(
    annual_cases=workflows_data["Invoice Approval"]["annual_volume"],
    manual_minutes=workflows_data["Invoice Approval"]["touch_time_min"],
    rework_rate=workflows_data["Invoice Approval"]["rework_rate"],
    rework_minutes=workflows_data["Invoice Approval"]["rework_time_min"],
    loaded_hourly_cost=workflows_data["Invoice Approval"]["hourly_cost"],
    expert_interruptions_per_case=workflows_data["Invoice Approval"]["escalation_rate"],
    expert_minutes_per_interruption=workflows_data["Invoice Approval"]["escalation_time_min"],
    expert_hourly_cost=workflows_data["Invoice Approval"]["expert_cost"],
)

print("INVOICE APPROVAL — Friction Analysis")
print("=" * 70)
print(f"Annual Volume: {workflows_data['Invoice Approval']['annual_volume']:,} invoices")
print()
print("COST BREAKDOWN:")
print(f"  Base Manual Cost:      €{invoice_friction['base_manual_cost']:,.0f}  (Necessary: review & approve)")
print(f"  Rework Cost:           €{invoice_friction['rework_cost']:,.0f}  (Preventable: missing fields)")
print(f"  Expert Interruption:   €{invoice_friction['expert_interruption_cost']:,.0f}  (Opportunity: manager approval)")
print(f"  ─────────────────────────────")
print(f"  Total Cost:            €{invoice_friction['total_cost']:,.0f}")
print()
print(f"  FRICTION (Rework + Expert): €{invoice_friction['friction_cost']:,.0f}")
print(f"  Friction %: {invoice_friction['friction_percentage']:.1f}%")
print()
print(f"Cost per invoice: €{invoice_friction['cost_per_case']:.2f}")
print()
print("INSIGHT: Only 6% of invoice cost is friction. But this is 4,200 invoices/year.")
print("In a 1M-invoice/year environment, €308 friction → €73k friction.")

---

## Part 3 — Worked Example 2: Support Triage

### Analysis: Where is Friction in Support?

**Workflow:**
- 18,000 support tickets per year
- Analyst triages each: 14 minutes (read ticket, categorize, route to right team)
- **Friction point 1 — Rework (8%):** Some tickets misrouted initially → must re-triage
  - 8% × 18 minutes per re-triage
- **Friction point 2 — Expert escalation (15%):** Complex issues or escalations → specialist handles → 30 minutes per case
  - 15% × 30 minutes specialist time

**Question:** Support is higher-volume and has more escalations than invoice. Is friction higher?

In [ ]:
# Worked Example 2: Support Triage Friction Analysis
support_friction = cost_of_friction(
    annual_cases=workflows_data["Support Triage"]["annual_volume"],
    manual_minutes=workflows_data["Support Triage"]["touch_time_min"],
    rework_rate=workflows_data["Support Triage"]["rework_rate"],
    rework_minutes=workflows_data["Support Triage"]["rework_time_min"],
    loaded_hourly_cost=workflows_data["Support Triage"]["hourly_cost"],
    expert_interruptions_per_case=workflows_data["Support Triage"]["escalation_rate"],
    expert_minutes_per_interruption=workflows_data["Support Triage"]["escalation_time_min"],
    expert_hourly_cost=workflows_data["Support Triage"]["expert_cost"],
)

print("SUPPORT TRIAGE — Friction Analysis")
print("=" * 70)
print(f"Annual Volume: {workflows_data['Support Triage']['annual_volume']:,} tickets")
print()
print("COST BREAKDOWN:")
print(f"  Base Manual Cost:      €{support_friction['base_manual_cost']:,.0f}  (Necessary: triage & route)")
print(f"  Rework Cost:           €{support_friction['rework_cost']:,.0f}  (Preventable: misrouting)")
print(f"  Expert Interruption:   €{support_friction['expert_interruption_cost']:,.0f}  (Opportunity: specialist time)")
print(f"  ─────────────────────────────")
print(f"  Total Cost:            €{support_friction['total_cost']:,.0f}")
print()
print(f"  FRICTION (Rework + Expert): €{support_friction['friction_cost']:,.0f}")
print(f"  Friction %: {support_friction['friction_percentage']:.1f}%")
print()
print(f"Cost per ticket: €{support_friction['cost_per_case']:.2f}")
print()
print("COMPARISON TO INVOICE:")
print(f"  Support friction % ({support_friction['friction_percentage']:.1f}%) is 4.5x higher than invoice (6%)")
print(f"  Expert time (€{support_friction['expert_interruption_cost']:,.0f}) is the biggest component")
print(f"  INSIGHT: High-friction workflow. This is where AI automation creates most value.")

---

## Part 4 — Worked Example 3: Management Reporting

### Analysis: Where is Friction in Reporting?

**Workflow:**
- 52 reports per year (monthly + quarterly)
- Analyst prepares each: 180 minutes (data gathering, analysis, formatting)
- **Friction point 1 — Rework (30%):** Reports require multiple revision cycles before approval
  - 30% × 120 minutes per revision
- **Friction point 2 — Expert escalation (5%):** Executive review and sign-off
  - 5% × 30 minutes executive time

**Question:** Knowledge work (reporting) often has high rework. How much friction is hidden?

In [ ]:
# Worked Example 3: Management Reporting Friction Analysis
reporting_friction = cost_of_friction(
    annual_cases=workflows_data["Management Reporting"]["annual_volume"],
    manual_minutes=workflows_data["Management Reporting"]["touch_time_min"],
    rework_rate=workflows_data["Management Reporting"]["rework_rate"],
    rework_minutes=workflows_data["Management Reporting"]["rework_time_min"],
    loaded_hourly_cost=workflows_data["Management Reporting"]["hourly_cost"],
    expert_interruptions_per_case=workflows_data["Management Reporting"]["escalation_rate"],
    expert_minutes_per_interruption=workflows_data["Management Reporting"]["escalation_time_min"],
    expert_hourly_cost=workflows_data["Management Reporting"]["expert_cost"],
)

print("MANAGEMENT REPORTING — Friction Analysis")
print("=" * 70)
print(f"Annual Volume: {workflows_data['Management Reporting']['annual_volume']} reports")
print()
print("COST BREAKDOWN:")
print(f"  Base Manual Cost:      €{reporting_friction['base_manual_cost']:,.0f}  (Necessary: data & analysis)")
print(f"  Rework Cost:           €{reporting_friction['rework_cost']:,.0f}  (Preventable: revisions)")
print(f"  Expert Interruption:   €{reporting_friction['expert_interruption_cost']:,.0f}  (Opportunity: executive review)")
print(f"  ─────────────────────────────")
print(f"  Total Cost:            €{reporting_friction['total_cost']:,.0f}")
print()
print(f"  FRICTION (Rework + Expert): €{reporting_friction['friction_cost']:,.0f}")
print(f"  Friction %: {reporting_friction['friction_percentage']:.1f}%")
print()
print(f"Cost per report: €{reporting_friction['cost_per_case']:.2f}")
print()
print("COMPARISON:")
print(f"  Reporting friction % ({reporting_friction['friction_percentage']:.1f}%) is HIGHEST of 3 workflows")
print(f"  Rework (€{reporting_friction['rework_cost']:,.0f}) is largest component (revision cycles)")
print(f"  INSIGHT: Knowledge work has high friction due to iterative revision cycles.")

---

## Part 5 — Comparison & Portfolio Opportunity

### Summary: Total Friction Across All 3 Workflows

In [ ]:
# Summary table: All 3 workflows comparison
summary_data = {
    "Workflow": ["Invoice Approval", "Support Triage", "Management Reporting"],
    "Volume": [4200, 18000, 52],
    "Base Cost (€)": [
        invoice_friction["base_manual_cost"],
        support_friction["base_manual_cost"],
        reporting_friction["base_manual_cost"],
    ],
    "Rework Cost (€)": [
        invoice_friction["rework_cost"],
        support_friction["rework_cost"],
        reporting_friction["rework_cost"],
    ],
    "Expert Cost (€)": [
        invoice_friction["expert_interruption_cost"],
        support_friction["expert_interruption_cost"],
        reporting_friction["expert_interruption_cost"],
    ],
    "Total Cost (€)": [
        invoice_friction["total_cost"],
        support_friction["total_cost"],
        reporting_friction["total_cost"],
    ],
    "Friction (€)": [
        invoice_friction["friction_cost"],
        support_friction["friction_cost"],
        reporting_friction["friction_cost"],
    ],
    "Friction %": [
        invoice_friction["friction_percentage"],
        support_friction["friction_percentage"],
        reporting_friction["friction_percentage"],
    ],
}

df_summary = pd.DataFrame(summary_data)

print("FRICTION ANALYSIS — All 3 Workflows")
print("=" * 100)
print(df_summary.to_string(index=False))
print("=" * 100)

total_base = df_summary["Base Cost (€)"].sum()
total_friction = df_summary["Friction (€)"].sum()
total_cost = df_summary["Total Cost (€)"].sum()
total_friction_pct = (total_friction / total_cost) * 100

print()
print(f"TOTAL BASE COST:    €{total_base:,.0f}  (Necessary work)")
print(f"TOTAL FRICTION:     €{total_friction:,.0f}  (Preventable + Opportunity)")
print(f"TOTAL COST:         €{total_cost:,.0f}")
print()
print(f"Friction as % of total: {total_friction_pct:.1f}%")
print()
print(f"KEY INSIGHT: €{total_friction:,.0f} in hidden friction across 3 workflows")
print(f"             This is 72% of the €19,319 labor savings from Lesson 03")
print(f"             Eliminating friction = capturing most of the value opportunity")

### Visualization: Where Does Friction Live?

In [ ]:
# Visualization: Stacked bar chart showing friction breakdown by workflow
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Stacked bar: Base vs Rework vs Expert
workflows = ["Invoice", "Support", "Reporting"]
base_costs = [invoice_friction["base_manual_cost"], support_friction["base_manual_cost"], reporting_friction["base_manual_cost"]]
rework_costs = [invoice_friction["rework_cost"], support_friction["rework_cost"], reporting_friction["rework_cost"]]
expert_costs = [invoice_friction["expert_interruption_cost"], support_friction["expert_interruption_cost"], reporting_friction["expert_interruption_cost"]]

x = np.arange(len(workflows))
width = 0.5

ax1.bar(x, base_costs, width, label="Base (Necessary)", color="#2ecc71")
ax1.bar(x, rework_costs, width, bottom=base_costs, label="Rework (Preventable)", color="#f39c12")
ax1.bar(x, expert_costs, width, bottom=np.array(base_costs) + np.array(rework_costs), label="Expert (Opportunity)", color="#e74c3c")

ax1.set_ylabel("Cost (€)", fontsize=11, fontweight="bold")
ax1.set_title("Workflow Cost Breakdown: Base vs Friction", fontsize=12, fontweight="bold")
ax1.set_xticks(x)
ax1.set_xticklabels(workflows)
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

# Pie chart: Friction % breakdown
friction_percentages = [invoice_friction["friction_percentage"], support_friction["friction_percentage"], reporting_friction["friction_percentage"]]
colors = ["#f39c12", "#e74c3c", "#3498db"]
ax2.pie(friction_percentages, labels=workflows, autopct="%1.1f%%", colors=colors, startangle=90)
ax2.set_title("Friction as % of Total Cost\n(Opportunity Ranking)", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

print("Chart shows:")
print("  LEFT: Stacked bars show composition (green=base, orange=rework, red=expert)")
print("  RIGHT: Friction % highlights opportunity (higher % = higher opportunity)")
print()
print("  Reporting (44% friction) → Highest opportunity for improvement")
print("  Support (27% friction) → High opportunity")
print("  Invoice (6% friction) → Lower opportunity at small scale")

---

## Part 6 — Sensitivity Analysis: What If Parameters Change?

### Scenario Analysis: How Does Friction Respond to Changes?

Friction is sensitive to rework rate and escalation rate. Small changes in process reliability create large swings in friction cost.

In [ ]:
# Sensitivity Analysis: Rework Rate Impact
print("SENSITIVITY ANALYSIS: Impact of Rework Rate Changes")
print("=" * 70)
print()

# Support triage: What if rework rate changes?
support_volume = workflows_data["Support Triage"]["annual_volume"]
rework_rates_test = [0.04, 0.06, 0.08, 0.10, 0.12]  # ±50% from baseline 8%

results = []
for rw_rate in rework_rates_test:
    result = cost_of_friction(
        annual_cases=support_volume,
        manual_minutes=workflows_data["Support Triage"]["touch_time_min"],
        rework_rate=rw_rate,
        rework_minutes=workflows_data["Support Triage"]["rework_time_min"],
        loaded_hourly_cost=workflows_data["Support Triage"]["hourly_cost"],
        expert_interruptions_per_case=workflows_data["Support Triage"]["escalation_rate"],
        expert_minutes_per_interruption=workflows_data["Support Triage"]["escalation_time_min"],
        expert_hourly_cost=workflows_data["Support Triage"]["expert_cost"],
    )
    results.append({
        "Rework Rate": f"{rw_rate*100:.0f}%",
        "Rework Cost (€)": f"€{result['rework_cost']:,.0f}",
        "Total Friction (€)": f"€{result['friction_cost']:,.0f}",
        "Friction %": f"{result['friction_percentage']:.1f}%",
    })

df_sensitivity = pd.DataFrame(results)
print("SUPPORT TRIAGE: Sensitivity to Rework Rate (Baseline: 8%)")
print(df_sensitivity.to_string(index=False))
print()
print("KEY INSIGHT: Doubling rework rate (8% → 16%) increases friction by €1,764")
print("             Small changes in routing accuracy have LARGE financial impact")
print()
print("IMPLICATION: If AI improves routing accuracy from 92% to 98%,")
print("             rework drops from 8% → 2% = €3,528 saved")

---

## Part 7 — Interactive Exercise

### Your Task: Analyze 4 New Workflows

Using the `cost_of_friction()` function, analyze friction in 4 additional workflows. Each represents a different friction type:

#### **Exercise 1: Missing Invoice Fields** (Data Quality Friction)
- **Workflow:** Data entry from supplier invoices (email PDFs)
- **Volume:** 15,000 per year
- **Touch time:** 8 minutes (data extraction)
- **Rework rate:** 15% (missing fields, wrong amounts)
- **Rework time:** 15 minutes each
- **Hourly cost:** €55 (clerk)
- **Escalation:** 30% escalated to analyst (5 min per escalation)
- **Expert cost:** €65/hr (analyst)

#### **Exercise 2: Support Escalations** (Routing Error Friction)
- **Workflow:** Customer support ticket routing
- **Volume:** 12,000 per year
- **Touch time:** 10 minutes (initial read & category)
- **Rework rate:** 18% (misrouted)
- **Rework time:** 15 minutes (re-route, notify customer)
- **Hourly cost:** €55 (support agent)
- **Escalation:** 25% escalated to senior agent (20 min each)
- **Expert cost:** €75/hr (senior agent)

#### **Exercise 3: Unclear Access Review Evidence** (Compliance Friction)
- **Workflow:** Access review for compliance (SOX/security)
- **Volume:** 1,000 per year
- **Touch time:** 15 minutes (review request, check policy)
- **Rework rate:** 20% (insufficient evidence, need follow-up)
- **Rework time:** 25 minutes (send follow-up, clarify)
- **Hourly cost:** €65 (compliance officer)
- **Escalation:** 20% escalated to auditor (60 min each for investigation)
- **Expert cost:** €90/hr (senior auditor)

#### **Exercise 4: Manual Policy Search** (Knowledge Fragmentation Friction)
- **Workflow:** Employee onboarding policy questions
- **Volume:** 200 per year (40 employees × ~5 questions each)
- **Touch time:** 40 minutes (manual search across policy docs)
- **Rework rate:** 25% (follow-up questions, clarifications needed)
- **Rework time:** 20 minutes (additional search, explanation)
- **Hourly cost:** €55 (HR team)
- **Escalation:** 15% escalated to manager (30 min each to explain/decide)
- **Expert cost:** €75/hr (manager)

**Your task:** Calculate friction for each exercise, then rank by friction % to identify the highest-priority opportunity for AI intervention.

In [ ]:
# Exercise: Learner analyzes 4 new workflows
# Provided data

exercises = {
    "Missing Invoice Fields": {
        "annual_volume": 15000,
        "touch_time_min": 8,
        "rework_rate": 0.15,
        "rework_time_min": 15,
        "hourly_cost": 55,
        "escalation_rate": 0.30,
        "escalation_time_min": 5,
        "expert_cost": 65,
    },
    "Support Escalations": {
        "annual_volume": 12000,
        "touch_time_min": 10,
        "rework_rate": 0.18,
        "rework_time_min": 15,
        "hourly_cost": 55,
        "escalation_rate": 0.25,
        "escalation_time_min": 20,
        "expert_cost": 75,
    },
    "Access Review Evidence": {
        "annual_volume": 1000,
        "touch_time_min": 15,
        "rework_rate": 0.20,
        "rework_time_min": 25,
        "hourly_cost": 65,
        "escalation_rate": 0.20,
        "escalation_time_min": 60,
        "expert_cost": 90,
    },
    "Manual Policy Search": {
        "annual_volume": 200,
        "touch_time_min": 40,
        "rework_rate": 0.25,
        "rework_time_min": 20,
        "hourly_cost": 55,
        "escalation_rate": 0.15,
        "escalation_time_min": 30,
        "expert_cost": 75,
    },
}

# Calculate friction for all 4 exercises
exercise_results = []

for workflow_name, params in exercises.items():
    result = cost_of_friction(
        annual_cases=params["annual_volume"],
        manual_minutes=params["touch_time_min"],
        rework_rate=params["rework_rate"],
        rework_minutes=params["rework_time_min"],
        loaded_hourly_cost=params["hourly_cost"],
        expert_interruptions_per_case=params["escalation_rate"],
        expert_minutes_per_interruption=params["escalation_time_min"],
        expert_hourly_cost=params["expert_cost"],
    )
    
    exercise_results.append({
        "Workflow": workflow_name,
        "Volume": params["annual_volume"],
        "Total Cost (€)": result["total_cost"],
        "Friction (€)": result["friction_cost"],
        "Friction %": result["friction_percentage"],
    })

df_exercises = pd.DataFrame(exercise_results).sort_values("Friction %", ascending=False)

print("EXERCISE RESULTS: 4 New Workflows Analyzed")
print("=" * 90)
print(df_exercises.to_string(index=False))
print("=" * 90)
print()
print("RANKING BY FRICTION %: Which workflows are highest priority?")
for idx, (_, row) in enumerate(df_exercises.iterrows(), 1):
    print(f"  {idx}. {row['Workflow']:25} {row['Friction %']:5.1f}% friction  (€{row['Friction (€)']:,.0f})")
print()
print("HIGHEST OPPORTUNITY: Manual Policy Search (73% friction)")
print("  → AI chatbot for policy Q&A would eliminate most of this friction")
print()
print("TOTAL EXERCISE FRICTION: €{:,.0f}".format(df_exercises["Friction (€)"].sum()))

---

## Part 8 — Lesson Completion & Transition

### Completion Checklist

**You're done with Lesson 06 when you:**

- [ ] Can explain the 3 friction layers (base, rework, expert) and give examples
- [ ] Calculated friction for invoice (€308), support (€9,603), and reporting (€4,022) workflows
- [ ] Understand that €13,933 total friction is 18% of total cost = opportunity
- [ ] Ranked 4 exercise workflows by friction % and identified highest-priority opportunity
- [ ] Can connect friction cost to L05 PE value levers (labor productivity)

---

### Key Insights Summary

**Lesson 06: Cost of Friction**

✓ **Friction = Preventable waste + Opportunity cost**
- Base cost (necessary) is not friction
- Rework (data quality issues, misrouting) is preventable waste
- Expert interruptions (escalations) are opportunity cost (expert pulled from other work)

✓ **Friction compounds at scale**
- 5% rework on 4,200 invoices = €231/year
- 5% rework on 1M invoices = €50k/year
- Scale matters for financial impact

✓ **Friction reveals opportunities**
- Reporting has 44% friction (revision cycles) → Focus here first
- Support has 27% friction (escalations) → High impact for AI routing
- Invoice has 6% friction (data quality) → Lower priority at small scale

✓ **Friction is sensitivity-driven**
- Small changes in rework rate → Large swings in cost
- Escalation rate has highest leverage (expert cost is high)
- Process improvements (routing accuracy, data quality) have huge ROI

---

## Next Lesson: Lesson 07 — Value Stream Metrics

**Lesson 06 measured friction:** "We have €13,933 in waste. Where is it?"

**Lesson 07 finds root causes:** "What creates this friction? Where are the bottlenecks?"

**Preview:** Value stream mapping translates friction into process metrics:
- Lead time (how long does a case take from start to finish?)
- Wait time (time spent waiting vs doing)
- First-pass yield (% of cases that don't need rework)
- Bottleneck step (which step creates the most delay?)

Then Lesson 08 will use **graphs to find bottlenecks systematically** — instead of guessing, we'll detect them automatically from process logs.